In [43]:
# Core
import json
import numpy as np
import pandas as pd
import joblib
import random

# Visualization (unchanged)
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# PyTorch (replaces TensorFlow/Keras)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import optuna

In [22]:
# Device setup (PyTorch equivalent of TF GPU check)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)


Using device: cuda
GPU Name: NVIDIA GeForce GTX 1650
CUDA Version: 13.0


In [23]:
scaler = joblib.load("../data/processed/X_scaler.pkl")
X_train = np.load("../data/processed/X_train.npy")
X_val = np.load("../data/processed/X_val.npy")
X_test = np.load("../data/processed/X_test.npy")
X_trainVal = np.load("../data/processed/X_trainVal.npy")

y_train = np.load("../data/processed/y_train.npy")
y_val = np.load("../data/processed/y_val.npy")
y_test = np.load("../data/processed/y_test.npy")
y_trainVal = np.load("../data/processed/y_trainVal.npy")
y_scaler = joblib.load("../data/processed/y_scaler.pkl")

with open("../data/processed/feature_names.json", "r") as f:
    feature_names = json.load(f)


In [24]:
class FinalParkinsonNN(nn.Module):
    def __init__(self, params, input_dim=18):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, params["units_l1"]), nn.ReLU(),
            nn.Linear(params["units_l1"], params["units_l2"]), nn.ReLU(),
            nn.Dropout(params["dropout"]),
            nn.Linear(params["units_l2"], params["units_l3"]), nn.ReLU(),
            nn.Linear(params["units_l3"], params["units_l4"]), nn.ReLU(),
            nn.Linear(params["units_l4"], 2)
        )

    def forward(self, x):
        return self.net(x)

In [25]:
with open("../models/best_nn_params.json", "r") as f:
    params = json.load(f)

In [74]:
nn_model = FinalParkinsonNN(params).to(DEVICE)
nn_model.load_state_dict(torch.load("../models/best_nn_model.pt"))
nn_model.eval()

rf_expert = joblib.load("../models/rf_expert.pkl")
ridge_expert = joblib.load("../models/ridge_expert.pkl")
enet_expert = joblib.load("../models/enet_expert.pkl")

In [35]:
# NN
X_tv_tensor = torch.tensor(X_trainVal, dtype=torch.float32).to(DEVICE)
with torch.no_grad():
    nn_preds_scaled = nn_model(X_tv_tensor).cpu().numpy()

# RF
rf_preds_scaled = rf_expert.predict(X_trainVal)

# Ridge
ridge_preds_scaled = ridge_expert.predict(X_trainVal)

# ENet
enet_preds_scaled = enet_expert.predict(X_trainVal)

# Stack
Z_trainVal = np.hstack([
    nn_preds_scaled,
    rf_preds_scaled,
    ridge_preds_scaled,
    enet_preds_scaled
])

print("Z_trainVal shape:", Z_trainVal.shape)

Z_trainVal shape: (4700, 8)


In [36]:
# NN
X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
with torch.no_grad():
    nn_val_scaled = nn_model(X_val_tensor).cpu().numpy()

rf_val_scaled = rf_expert.predict(X_val)
ridge_val_scaled = ridge_expert.predict(X_val)
enet_val_scaled = enet_expert.predict(X_val)

Z_val = np.hstack([
    nn_val_scaled,
    rf_val_scaled,
    ridge_val_scaled,
    enet_val_scaled
])

print("Z_val shape:", Z_val.shape)

Z_val shape: (940, 8)


In [ ]:
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)

with torch.no_grad():
    nn_test_scaled = nn_model(X_test_tensor).cpu().numpy()

rf_test_scaled = rf_expert.predict(X_test)
ridge_test_scaled = ridge_expert.predict(X_test)
enet_test_scaled = enet_expert.predict(X_test)

Z_test = np.hstack([
    nn_test_scaled,
    rf_test_scaled,
    ridge_test_scaled,
    enet_test_scaled
])

print("Z_test shape:", Z_test.shape)

In [ ]:
np.save("../data/interim/Z_trainVal.npy", Z_trainVal)
np.save("../data/interim/Z_val.npy", Z_val)
np.save("../data/interim/Z_test.npy", Z_test)

print("✅ Z splits exported.")

✅ Z splits exported.


/e/conda/envs/ml/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [38]:
class MoEGate(nn.Module):
    def __init__(self, input_dim=8, hidden_dim=32, n_experts=4):
        super().__init__()
        
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        
        # Separate heads for motor and total
        self.motor_head = nn.Linear(hidden_dim, n_experts)
        self.total_head = nn.Linear(hidden_dim, n_experts)

    def forward(self, z):
        x = self.shared(z)

        motor_weights = F.softmax(self.motor_head(x), dim=1)
        total_weights = F.softmax(self.total_head(x), dim=1)

        return motor_weights, total_weights

In [39]:
def moe_forward(gate, z, n_experts=4):
    
    motor_w, total_w = gate(z)

    # Split expert predictions
    experts = z.view(-1, n_experts, 2)

    motor_preds = experts[:, :, 0]
    total_preds = experts[:, :, 1]

    motor_out = torch.sum(motor_w * motor_preds, dim=1)
    total_out = torch.sum(total_w * total_preds, dim=1)

    return torch.stack([motor_out, total_out], dim=1)

In [40]:
class GateDataset(torch.utils.data.Dataset):
    def __init__(self, Z, y):
        self.Z = torch.tensor(Z, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.Z)

    def __getitem__(self, idx):
        return self.Z[idx], self.y[idx]

In [41]:
train_ds = GateDataset(Z_trainVal, y_trainVal)
val_ds   = GateDataset(Z_val, y_val)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_ds, batch_size=64)

gate_model = MoEGate(input_dim=8, hidden_dim=32, n_experts=4).to(DEVICE)

In [69]:
optimizer = torch.optim.Adam(gate_model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

best_val = float("inf")
patience = 50
patience_ctr = 0

for epoch in range(300):
    gate_model.train()
    train_loss = 0

    for z_batch, y_batch in train_loader:
        z_batch = z_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()

        motor_w, total_w = gate_model(z_batch)

        experts = z_batch.view(-1, 4, 2)

        motor_preds = experts[:, :, 0]
        total_preds = experts[:, :, 1]

        motor_out = torch.sum(motor_w * motor_preds, dim=1)
        total_out = torch.sum(total_w * total_preds, dim=1)

        preds = torch.stack([motor_out, total_out], dim=1)

        mse_loss = criterion(preds, y_batch)

        # 🔥 Entropy regularization
        entropy_motor = -torch.sum(motor_w * torch.log(motor_w + 1e-8), dim=1).mean()
        entropy_total = -torch.sum(total_w * torch.log(total_w + 1e-8), dim=1).mean()

        entropy_loss = -(entropy_motor + entropy_total)

        lambda_entropy = 0.01

        loss = mse_loss + lambda_entropy * entropy_loss

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    gate_model.eval()
    val_loss = 0

    with torch.no_grad():
        for z_batch, y_batch in val_loader:
            z_batch = z_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            preds = moe_forward(gate_model, z_batch)
            val_loss += criterion(preds, y_batch).item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1:03d} | Train MSE: {train_loss:.6f} | Val MSE: {val_loss:.6f}")

    if val_loss < best_val:
        best_val = val_loss
        patience_ctr = 0
        torch.save(gate_model.state_dict(), "../models/best_gate.pt")
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print("Early stopping.")
            break

Epoch 001 | Train MSE: 0.006038 | Val MSE: 0.010335
Epoch 002 | Train MSE: 0.005705 | Val MSE: 0.010981
Epoch 003 | Train MSE: 0.004774 | Val MSE: 0.011938
Epoch 004 | Train MSE: 0.003444 | Val MSE: 0.011808
Epoch 005 | Train MSE: 0.002842 | Val MSE: 0.013885
Epoch 006 | Train MSE: 0.003188 | Val MSE: 0.011604
Epoch 007 | Train MSE: 0.002672 | Val MSE: 0.011261
Epoch 008 | Train MSE: 0.002358 | Val MSE: 0.012175
Epoch 009 | Train MSE: 0.001697 | Val MSE: 0.012271
Epoch 010 | Train MSE: 0.000676 | Val MSE: 0.011658
Epoch 011 | Train MSE: -0.001394 | Val MSE: 0.013338
Epoch 012 | Train MSE: -0.002508 | Val MSE: 0.012833
Epoch 013 | Train MSE: -0.001130 | Val MSE: 0.013963
Epoch 014 | Train MSE: -0.003035 | Val MSE: 0.012308
Epoch 015 | Train MSE: -0.003265 | Val MSE: 0.012662
Epoch 016 | Train MSE: -0.002912 | Val MSE: 0.012787
Epoch 017 | Train MSE: -0.003325 | Val MSE: 0.012822
Epoch 018 | Train MSE: -0.002901 | Val MSE: 0.013350
Epoch 019 | Train MSE: -0.003158 | Val MSE: 0.013813
Epo

In [70]:
gate_model.load_state_dict(torch.load("../models/best_gate.pt"))
gate_model.eval()

Z_test_tensor = torch.tensor(Z_test, dtype=torch.float32).to(DEVICE)

with torch.no_grad():
    preds_scaled = moe_forward(gate_model, Z_test_tensor).cpu().numpy()

# Inverse scale FINAL output
preds = y_scaler.inverse_transform(preds_scaled)

mse = mean_squared_error(y_test, preds)
r2  = r2_score(y_test, preds)

print(f"MoE Test MSE: {mse:.4f}")
print(f"MoE Test R2 : {r2:.4f}")

MoE Test MSE: 7.4710
MoE Test R2 : 0.9133


In [71]:
with torch.no_grad():
    motor_w, total_w = gate_model(Z_test_tensor)

print("Mean motor weights:", motor_w.mean(dim=0))
print("Mean total weights:", total_w.mean(dim=0))

Mean motor weights: tensor([0.0028, 0.9244, 0.0348, 0.0380], device='cuda:0')
Mean total weights: tensor([0.0013, 0.8768, 0.0606, 0.0613], device='cuda:0')


In [72]:
torch.save({
    "model_state_dict": gate_model.state_dict(),
    "input_dim": Z_train.shape[1],
}, "../models/best_gate_full.pt")